## Linear Programming: Produce Inventory To Meet Demand

A company produces and sells a product. On any given day, there is some demand that needs to be met, it costs some amount of money per product produced (if any) and some amount of product can be carried over to the next day. Each item carried over incurs some inventory cost. The revenue is fixed and we're trying to minimize the costs.

This problem can be formulated as follows:

$$
\begin{array}{rl}
\min\quad & \displaystyle \sum_{i=1}^D \left( p_ix_i + Cc_i \right) \\
\text{s.t.}\quad
& \left.
    \begin{array}{l}
      x_i + c_{i-1} - c_{i} = d_i \\
      x_i, c_i \ge 0
    \end{array}
  \right\} \quad \forall i \in \{1, \dots, D\} \\
& c_0 = 0
\end{array}
$$

with variables defined as follows:

- $x_i$: the number of units to produce on day $i$
- $p_i$: the cost-per-unit of production on day $i$
- $c_i$: the number of products carried over on day $i$
- $C$: the inventory cost per product
- $D$: the number of days

Problem described in 2.8 of [Operations Research (1): Models and Applications](https://www.coursera.org/learn/operations-research-modeling/home/welcome).


In [2]:
# Imports and input data

import numpy as np
import pandas as pd
from scipy.optimize import linprog

day_demand = [100, 150, 200, 170]
production_cost = [9, 12, 10, 12]
inventory_cost = 1
days = len(day_demand)

In [3]:
# Set objective and constraints

# Variables: [day 1-n produce, carry over from day 0-(n-1)]

# Flip coef sign to turn maximization problem into minimization problem
coef = np.array(production_cost + [inventory_cost] * days)

# Day i production + carry over from day i-1 = day i demand
# So the left-hand size is just 2 concatenated identity matrices
A_eq = np.tile(np.identity(days), 2)
b_eq = day_demand

# Day i production + carry over from day i-1 = day i demand + carry over from day i
# Thus day i production + carry over from day i-1 - carry over from day i = day i demand
# The left-hand side ends up being:
# [1 0 0 0  1 -1  0  0]
# [0 1 0 0  0  1 -1  0]
# [0 0 1 0  0  0  1 -1]
# [0 0 0 1  0  0  0  1]
rolled = np.roll(np.identity(days), 1)
rolled[:, 0] = 0
A_eq = np.concat([np.identity(days), np.identity(days) - rolled], axis=1)
b_eq = np.array(day_demand)

bounds = [[0, None] for _ in range(days * 2)]
# Day 0 cannot have any carry-over
bounds[days][1] = 0

In [4]:
# Run linear programming
res = linprog(coef, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 6270.0
              x: [ 2.500e+02  0.000e+00  3.700e+02  0.000e+00  0.000e+00
                   1.500e+02  0.000e+00  1.700e+02]
            nit: 0
          lower:  residual: [ 2.500e+02  0.000e+00  3.700e+02  0.000e+00
                              0.000e+00  1.500e+02  0.000e+00  1.700e+02]
                 marginals: [ 0.000e+00  2.000e+00  0.000e+00  1.000e+00
                              0.000e+00  0.000e+00  1.000e+00  0.000e+00]
          upper:  residual: [       inf        inf        inf        inf
                              0.000e+00        inf        inf        inf]
                 marginals: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00
                             -8.000e+00  0.000e+00  0.000e+00  0.000e+00]
          eqlin:  residual: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00]
                 marginals: [ 9.000e+00  1.000e+

In [5]:
# Print solution details

print("Optimal strategy:")

df = pd.DataFrame({
    "demand": day_demand,
    "produced": res.x[:days],
    # Day 0 carry-over wraps around to day 4 carry-over, both are 0
    "carried_over": np.roll(res.x[days:], -1)
}, index=pd.Series(range(1, days+1)))

df["inventory_cost"] = df.carried_over * inventory_cost
df["production_cost"] = df.produced * production_cost
df["total_cost"] = df.inventory_cost + df.production_cost

df = pd.concat([df, df.sum().to_frame("sum").T])

df.index.name = "day"

df = df.map(int)

sum_row = pd.IndexSlice[df.index[df.index == "sum"], :]
df.style.map(lambda _: "font-weight: bold", subset=sum_row)

Optimal strategy:


,demand,produced,carried_over,inventory_cost,production_cost,total_cost
day,,,,,,
1,100,250,150,150,2250,2400
2,150,0,0,0,0,0
3,200,370,170,170,3700,3870
4,170,0,0,0,0,0
sum,620,620,320,320,5950,6270
